# 🌳 Digital Forest Twin - Tree Data Importer (Python)

This notebook provides a step-by-step workflow to import tree data from CSV files into the Digital Forest Twin database.

## Workflow Overview

1. **Setup**: Load dependencies and connect to database
2. **Introspect**: Display available database schema
3. **Load Reference Data**: Explore species, locations, and other lookup tables
4. **Load CSV**: Load your data file
5. **Interactive Column Mapping**: Match CSV columns to database columns using widgets
6. **Lookup Validation**: Automatically check if categorical data matches database lookup tables
7. **Value Mapping**: Interactively map unmatched values to existing DB entries or create new ones
8. **Transform & Preview**: Review transformed data before insertion
9. **Execute Import**: Dry-run first, then actual import

## Key Features

- **Interactive Widgets**: Use dropdowns to map columns without writing code
- **Lookup Table Detection**: Automatically identifies which columns reference lookup tables
- **Validation**: Checks if CSV values exist in database before import
- **Custom Mappings**: Create mappings for species codes, location names, etc.
- **New Entry Creation**: Optionally add new entries to lookup tables
- **Configuration Saving**: Save mappings as JSON for reuse with similar files

## Instructions

- Run cells sequentially from top to bottom
- The interactive widgets will guide you through the mapping process
- Configuration is automatically saved and can be reloaded for future imports

## Step 1: Setup - Load Dependencies and Connect to Database

In [1]:
# Install/import required packages
import os
import psycopg2
from pathlib import Path
from typing import Any, Dict, List, Optional
import json

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Interactive widgets for column mapping
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML

    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("⚠️  ipywidgets not installed. Install with: pip install ipywidgets")
    print("   The interactive mapping features will not work without it.")

try:
    from pyproj import Transformer

    HAS_PYPROJ = True
except ImportError:
    HAS_PYPROJ = False
    print("⚠️  pyproj not installed. Install with: pip install pyproj")

# Load environment
load_dotenv("../docker/.env")
print("✓ Dependencies loaded")

/var/folders/9j/h625g8x90hd0_p1jpflmxd3h0000gn/T/ipykernel_24046/554175464.py:8: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


✓ Dependencies loaded


In [2]:
# Configure database connection
from pathlib import Path

# Only change directory if not already in docker folder
if not os.getcwd().endswith("docker"):
    target_dir = Path(os.getcwd())
    # Navigate up to find the project root
    while target_dir.name != "digital-twin" and target_dir.parent != target_dir:
        target_dir = target_dir.parent
    target_dir = target_dir / "docker"
    if target_dir.exists():
        os.chdir(target_dir)

print(f"Working directory: {os.getcwd()}")

# Database connection settings
POSTGRES_HOST = "localhost"
POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DATABASE = os.getenv("POSTGRES_DB", "postgres")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")

# Supavisor pooler requires tenant ID in the username format: user.tenant_id
POOLER_TENANT_ID = os.getenv("POOLER_TENANT_ID", "")
POSTGRES_USER_POOLER = (
    f"{POSTGRES_USER}.{POOLER_TENANT_ID}" if POOLER_TENANT_ID else POSTGRES_USER
)

if not POSTGRES_PASSWORD:
    print("❌ POSTGRES_PASSWORD not found in docker/.env")
else:
    print(
        f"✓ Database config: {POSTGRES_USER_POOLER}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}"
    )

# Create SQLAlchemy engine for pandas (avoids deprecation warnings)
from urllib.parse import quote_plus

encoded_password = quote_plus(POSTGRES_PASSWORD) if POSTGRES_PASSWORD else ""
encoded_user = quote_plus(POSTGRES_USER_POOLER)
DATABASE_URL = f"postgresql://{encoded_user}:{encoded_password}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}"
engine = create_engine(DATABASE_URL)

# Test connection via Supavisor pooler
try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        user=POSTGRES_USER_POOLER,
        password=POSTGRES_PASSWORD,
        database=POSTGRES_DATABASE,
        port=POSTGRES_PORT,
    )
    conn.close()
    print("✓ Database connection successful")
except Exception as e:
    print(f"❌ Database connection failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Ensure Docker containers are running: docker compose up -d")
    print("  2. Check Supavisor status: docker ps --filter 'name=dftdb-pooler'")

Working directory: /Users/maximiliansperlich/Developer/work/digital-twin/docker
✓ Database config: postgres.digital-forest-twin-local@localhost:5432/postgres
✓ Database connection successful


## Step 2: Introspect Database Schema

In [3]:
def introspect_schema():
    """Query database schema from information_schema"""
    query = """
        SELECT
            table_schema,
            table_name,
            column_name
        FROM information_schema.columns
        WHERE table_schema IN ('shared', 'trees', 'sensor', 'pointclouds', 'environments')
        ORDER BY table_schema, table_name, ordinal_position
    """

    try:
        conn = psycopg2.connect(
            host=POSTGRES_HOST,
            user=POSTGRES_USER_POOLER,
            password=POSTGRES_PASSWORD,
            database=POSTGRES_DATABASE,
            port=POSTGRES_PORT,
        )
        df = pd.read_sql(query, conn)
        conn.close()

        # Build nested dictionary
        schema_info = {}
        for _, row in df.iterrows():
            schema = row["table_schema"]
            table = row["table_name"]
            column = row["column_name"]

            if schema not in schema_info:
                schema_info[schema] = {}
            if table not in schema_info[schema]:
                schema_info[schema][table] = []

            schema_info[schema][table].append(column)

        return schema_info
    except Exception as e:
        print(f"❌ Schema introspection failed: {e}")
        return {}


# Get schema
schema_info = introspect_schema()
print(
    f"✓ Found schema info for {sum(len(tables) for tables in schema_info.values())} tables"
)

✓ Found schema info for 41 tables


/var/folders/9j/h625g8x90hd0_p1jpflmxd3h0000gn/T/ipykernel_24046/1365331066.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [4]:
# Display schema
print("\n" + "=" * 80)
print("DATABASE SCHEMA - Available Tables & Columns")
print("=" * 80)

for schema_name in sorted(schema_info.keys()):
    print(f"\n📦 Schema: {schema_name}")
    tables = schema_info[schema_name]
    for table_name in sorted(tables.keys()):
        columns = tables[table_name]
        print(f"  📋 {table_name} ({len(columns)} columns)")
        for i, col in enumerate(columns, 1):
            print(f"     {i:2d}. {col}")


DATABASE SCHEMA - Available Tables & Columns

📦 Schema: environments
  📋 active_environments (34 columns)
      1. variantid
      2. parentvariantid
      3. locationid
      4. scenarioid
      5. varianttypeid
      6. processid
      7. variantname
      8. startdate
      9. enddate
     10. avgtemperature_c
     11. avghumidity_percent
     12. totalprecipitation_mm
     13. avgglobalradiation
     14. avgco2_ppm
     15. avgwindspeed_ms
     16. dominantwinddirection_deg
     17. avgsoilmoisture_percent
     18. avgsoiltemperature_c
     19. soilph
     20. nutrientnitrogen_mg_kg
     21. nutrientphosphorus_mg_kg
     22. nutrientpotassium_mg_kg
     23. stressfactor
     24. description
     25. researchnotes
     26. createdat
     27. updatedat
     28. createdby
     29. updatedby
     30. duration_days
     31. is_active
     32. locationname
     33. scenarioname
     34. varianttypename
  📋 environments (29 columns)
      1. variantid
      2. parentvariantid
      3. lo

## Step 3: Load Reference Data

In [5]:
def load_reference_data():
    """Load reference tables for mapping help"""
    reference_data = {}

    try:
        # Load Species
        reference_data["Species"] = pd.read_sql(
            "SELECT SpeciesID, CommonName, ScientificName FROM shared.Species ORDER BY CommonName",
            engine,
        )

        # Load Locations
        reference_data["Locations"] = pd.read_sql(
            "SELECT LocationID, LocationName FROM shared.Locations ORDER BY LocationName",
            engine,
        )

        # Try to load SensorTypes
        try:
            reference_data["SensorTypes"] = pd.read_sql(
                "SELECT SensorTypeID, SensorTypeName FROM sensor.SensorTypes ORDER BY SensorTypeName",
                engine,
            )
        except:
            pass

        return reference_data
    except Exception as e:
        print(f"❌ Failed to load reference data: {e}")
        return {}


# Load reference data
reference_data = load_reference_data()
print(f"✓ Loaded reference data: {', '.join(reference_data.keys())}")

✓ Loaded reference data: Species, Locations, SensorTypes


In [6]:
# Display reference data
print("\n" + "=" * 80)
print("📚 REFERENCE DATA - Use for mapping CSV values to database IDs")
print("=" * 80)

if "Species" in reference_data:
    print("\n📚 Species:")
    display(reference_data["Species"])

if "Locations" in reference_data:
    print("\n📚 Locations:")
    display(reference_data["Locations"])

if "SensorTypes" in reference_data:
    print("\n📚 SensorTypes:")
    display(reference_data["SensorTypes"])


📚 REFERENCE DATA - Use for mapping CSV values to database IDs

📚 Species:


,speciesid,commonname,scientificname
0,6,Douglas Fir,Pseudotsuga menziesii
1,1,European Beech,Fagus sylvatica
2,3,Norway Spruce,Picea abies
3,2,Pedunculate Oak,Quercus robur
4,5,Scots Pine,Pinus sylvestris
5,4,Silver Fir,Abies alba



📚 Locations:


,locationid,locationname
0,3,Black Forest Test Site
1,4,Ecosense_MixedPlot
2,1,University Forest Plot A
3,2,University Forest Plot B



📚 SensorTypes:


,sensortypeid,sensortypename
0,9,Barometric_Pressure
1,3,CO2
2,2,Humidity
3,12,Leaf_Wetness
4,4,Light
5,8,Precipitation
6,13,Sap_Flow
7,5,Soil_Moisture
8,11,Soil_Temperature
9,10,Solar_Radiation


## Interactive Data Import Wizard

⚠️ **Prerequisites**: Make sure you have:
1. Run cells 1-4 above (setup, database connection, schema introspection, reference data)
2. Run the "Load CSV File" section below (Step 4) to load your CSV

This wizard provides an interactive approach to:
1. **Column Mapping**: Match CSV columns to database columns using dropdowns
2. **Lookup Validation**: Automatically check if categorical data matches database lookup tables
3. **Custom Mapping**: Create mappings for values that don't exist in the database
4. **New Entry Creation**: Optionally add missing entries to lookup tables
5. **Preview & Import**: Review transformed data and execute import

In [12]:
# ============================================================================
# INTERACTIVE COLUMN MAPPING & LOOKUP VALIDATION SYSTEM
# ============================================================================

# Check prerequisites
prerequisites_ok = True
if "df" not in dir() or df is None:
    print("❌ CSV not loaded! Run the 'Load CSV File' cell first.")
    prerequisites_ok = False
if "schema_info" not in dir() or not schema_info:
    print("❌ Schema info not loaded! Run the 'Introspect Database Schema' cell first.")
    prerequisites_ok = False
if "reference_data" not in dir():
    print(
        "⚠️  Reference data not loaded. Run the 'Load Reference Data' cell for better matching."
    )

if not prerequisites_ok:
    raise RuntimeError(
        "Prerequisites not met. Please run the required cells above first."
    )

from dataclasses import dataclass, field


@dataclass
class LookupTableConfig:
    """Configuration for a database lookup table"""

    table_name: str  # e.g., "shared.Species"
    id_column: str  # e.g., "SpeciesID"
    display_columns: (
        list  # Columns to show for matching, e.g., ["CommonName", "ScientificName"]
    )
    target_column: str  # The column in target table that references this lookup


@dataclass
class ColumnMappingResult:
    """Result of column mapping process"""

    csv_column: str
    db_schema: str
    db_table: str
    db_column: str
    is_lookup: bool = False
    lookup_config: LookupTableConfig = None
    value_mappings: dict = field(default_factory=dict)  # CSV value -> DB ID


# Define known lookup tables and their configurations
LOOKUP_TABLES = {
    "shared.species": LookupTableConfig(
        table_name="shared.Species",
        id_column="SpeciesID",
        display_columns=["CommonName", "ScientificName"],
        target_column="SpeciesID",
    ),
    "shared.locations": LookupTableConfig(
        table_name="shared.Locations",
        id_column="LocationID",
        display_columns=["LocationName"],
        target_column="LocationID",
    ),
    "trees.treestatus": LookupTableConfig(
        table_name="trees.TreeStatus",
        id_column="TreeStatusID",
        display_columns=["TreeStatusName"],
        target_column="TreeStatusID",
    ),
    "trees.tapertypes": LookupTableConfig(
        table_name="trees.TaperTypes",
        id_column="TaperTypeID",
        display_columns=["TaperTypeName"],
        target_column="TaperTypeID",
    ),
    "trees.straightnesstypes": LookupTableConfig(
        table_name="trees.StraightnessTypes",
        id_column="StraightnessTypeID",
        display_columns=["StraightnessName"],
        target_column="StraightnessTypeID",
    ),
    "trees.branchingpatterns": LookupTableConfig(
        table_name="trees.BranchingPatterns",
        id_column="BranchingPatternID",
        display_columns=["BranchingPatternName"],
        target_column="BranchingPatternID",
    ),
    "trees.barkcharacteristics": LookupTableConfig(
        table_name="trees.BarkCharacteristics",
        id_column="BarkCharacteristicID",
        display_columns=["BarkCharacteristicName"],
        target_column="BarkCharacteristicID",
    ),
    "shared.varianttypes": LookupTableConfig(
        table_name="shared.VariantTypes",
        id_column="VariantTypeID",
        display_columns=["VariantTypeName"],
        target_column="VariantTypeID",
    ),
    "shared.scenarios": LookupTableConfig(
        table_name="shared.Scenarios",
        id_column="ScenarioID",
        display_columns=["ScenarioName"],
        target_column="ScenarioID",
    ),
    "shared.soiltypes": LookupTableConfig(
        table_name="shared.SoilTypes",
        id_column="SoilTypeID",
        display_columns=["SoilTypeName"],
        target_column="SoilTypeID",
    ),
    "shared.climatezones": LookupTableConfig(
        table_name="shared.ClimateZones",
        id_column="ClimateZoneID",
        display_columns=["ClimateZoneName"],
        target_column="ClimateZoneID",
    ),
}

# Columns in target tables that reference lookup tables
LOOKUP_COLUMN_MAPPINGS = {
    "trees.trees.speciesid": "shared.species",
    "trees.trees.locationid": "shared.locations",
    "trees.trees.treestatusid": "trees.treestatus",
    "trees.trees.branchingpatternid": "trees.branchingpatterns",
    "trees.trees.barkcharacteristicid": "trees.barkcharacteristics",
    "trees.trees.varianttypeid": "shared.varianttypes",
    "trees.trees.scenarioid": "shared.scenarios",
    "trees.stems.tapertypeid": "trees.tapertypes",
    "trees.stems.straightnesstypeid": "trees.straightnesstypes",
    "shared.locations.soiltypeid": "shared.soiltypes",
    "shared.locations.climatezoneid": "shared.climatezones",
}

print("✓ Lookup table configurations loaded")
print(f"  Known lookup tables: {len(LOOKUP_TABLES)}")
print(f"  Column-to-lookup mappings: {len(LOOKUP_COLUMN_MAPPINGS)}")

✓ Lookup table configurations loaded
  Known lookup tables: 11
  Column-to-lookup mappings: 11


In [ ]:
# ============================================================================
# STEP 1: Interactive Column Mapping
# ============================================================================


class InteractiveColumnMapper:
    """Interactive widget-based column mapper"""

    def __init__(self, csv_df: pd.DataFrame, schema_info: dict):
        self.csv_df = csv_df
        self.schema_info = schema_info
        self.mappings = {}
        self.output = widgets.Output()

        # Build flat list of db columns
        self.db_columns = self._build_db_column_list()

    def _build_db_column_list(self) -> list:
        """Build flat list of schema.table.column options"""
        options = ["-- SKIP --", "-- COORDINATE (lat/lon) --", "-- COORDINATE (x/y) --"]
        for schema, tables in sorted(self.schema_info.items()):
            for table, columns in sorted(tables.items()):
                for col in columns:
                    options.append(f"{schema}.{table}.{col}")
        return options

    def _get_sample_values(self, column: str, n: int = 5) -> str:
        """Get sample values from CSV column"""
        unique = self.csv_df[column].dropna().unique()
        samples = [str(v) for v in unique[:n]]
        if len(unique) > n:
            samples.append(f"... +{len(unique)-n} more")
        return ", ".join(samples)

    def display(self):
        """Display interactive mapping interface"""
        print("\n" + "=" * 80)
        print("📋 STEP 1: Column Mapping")
        print("=" * 80)
        print("Match each CSV column to a database column, or SKIP if not needed.\n")

        self.dropdowns = {}
        rows = []

        for csv_col in self.csv_df.columns:
            # Create dropdown for this column
            dropdown = widgets.Dropdown(
                options=self.db_columns,
                value="-- SKIP --",
                layout=widgets.Layout(width="350px"),
            )
            self.dropdowns[csv_col] = dropdown

            # Sample values
            samples = self._get_sample_values(csv_col)

            # Create row
            label = widgets.HTML(
                value=f"<b>{csv_col}</b><br><small style='color:gray'>{samples[:60]}...</small>",
                layout=widgets.Layout(width="250px"),
            )
            arrow = widgets.HTML(value="→", layout=widgets.Layout(width="30px"))

            row = widgets.HBox([label, arrow, dropdown])
            rows.append(row)

        # Confirm button
        confirm_btn = widgets.Button(
            description="✓ Confirm Mappings",
            button_style="success",
            layout=widgets.Layout(width="200px", margin="20px 0"),
        )
        confirm_btn.on_click(self._on_confirm)

        # Display all
        display(widgets.VBox(rows))
        display(confirm_btn)
        display(self.output)

    def _on_confirm(self, btn):
        """Handle confirm button click"""
        with self.output:
            clear_output()
            self.mappings = {}

            print("\n" + "-" * 40)
            print("✓ Column Mappings Confirmed:")
            print("-" * 40)

            for csv_col, dropdown in self.dropdowns.items():
                target = dropdown.value
                if target not in ["-- SKIP --"]:
                    self.mappings[csv_col] = target
                    print(f"  {csv_col} → {target}")

            print(
                f"\n📊 {len(self.mappings)} columns mapped, {len(self.csv_df.columns) - len(self.mappings)} skipped"
            )
            print("\n✓ Run the next cell to validate lookup columns")

    def get_mappings(self) -> dict:
        """Return the confirmed mappings"""
        return self.mappings


# Create and display mapper
column_mapper = InteractiveColumnMapper(df, schema_info)
column_mapper.display()


📋 STEP 1: Column Mapping
Match each CSV column to a database column, or SKIP if not needed.



Button(button_style='success', description='✓ Confirm Mappings', layout=Layout(margin='20px 0', width='200px')…

Output()

In [14]:
# ============================================================================
# STEP 2: Lookup Table Validation
# ============================================================================


class LookupValidator:
    """Validates CSV values against database lookup tables"""

    def __init__(self, mappings: dict, csv_df: pd.DataFrame):
        self.mappings = mappings
        self.csv_df = csv_df
        self.lookup_issues = {}  # csv_col -> {csv_value: None (unmatched)}
        self.lookup_data = {}  # lookup_table -> DataFrame
        self.value_mappings = {}  # csv_col -> {csv_value: db_id}

    def load_lookup_data(self):
        """Load all relevant lookup tables from database"""
        conn = psycopg2.connect(
            host=POSTGRES_HOST,
            user=POSTGRES_USER_POOLER,
            password=POSTGRES_PASSWORD,
            database=POSTGRES_DATABASE,
            port=POSTGRES_PORT,
        )

        for mapping_key, lookup_key in LOOKUP_COLUMN_MAPPINGS.items():
            if lookup_key not in self.lookup_data:
                config = LOOKUP_TABLES[lookup_key]
                cols = [config.id_column] + config.display_columns
                query = f"SELECT {', '.join(cols)} FROM {config.table_name}"
                try:
                    self.lookup_data[lookup_key] = pd.read_sql(query, conn)
                except Exception as e:
                    print(f"⚠️ Could not load {config.table_name}: {e}")

        conn.close()
        print(f"✓ Loaded {len(self.lookup_data)} lookup tables")

    def identify_lookup_columns(self) -> list:
        """Identify which mapped columns reference lookup tables"""
        lookup_columns = []

        for csv_col, db_target in self.mappings.items():
            if db_target.startswith("--"):
                continue

            # Check if this column references a lookup table
            target_lower = db_target.lower()
            if target_lower in LOOKUP_COLUMN_MAPPINGS:
                lookup_key = LOOKUP_COLUMN_MAPPINGS[target_lower]
                lookup_columns.append(
                    {
                        "csv_column": csv_col,
                        "db_target": db_target,
                        "lookup_key": lookup_key,
                        "lookup_config": LOOKUP_TABLES[lookup_key],
                    }
                )

        return lookup_columns

    def validate_values(self, lookup_columns: list) -> dict:
        """Check if CSV values exist in lookup tables"""
        validation_results = {}

        for lc in lookup_columns:
            csv_col = lc["csv_column"]
            lookup_key = lc["lookup_key"]
            config = lc["lookup_config"]

            if lookup_key not in self.lookup_data:
                continue

            lookup_df = self.lookup_data[lookup_key]
            csv_values = self.csv_df[csv_col].dropna().unique()

            # Get all possible match values from lookup table
            all_lookup_values = set()
            for disp_col in config.display_columns:
                if disp_col in lookup_df.columns:
                    all_lookup_values.update(
                        lookup_df[disp_col].dropna().astype(str).str.lower()
                    )

            # Check each CSV value
            matched = []
            unmatched = []

            for csv_val in csv_values:
                csv_val_str = str(csv_val).strip()
                csv_val_lower = csv_val_str.lower()

                # Try exact match first
                if csv_val_lower in all_lookup_values:
                    matched.append(csv_val_str)
                else:
                    # Try partial match
                    partial_match = None
                    for lv in all_lookup_values:
                        if csv_val_lower in lv or lv in csv_val_lower:
                            partial_match = lv
                            break

                    if partial_match:
                        matched.append((csv_val_str, f"partial: {partial_match}"))
                    else:
                        unmatched.append(csv_val_str)

            validation_results[csv_col] = {
                "lookup_key": lookup_key,
                "lookup_config": config,
                "lookup_df": lookup_df,
                "matched": matched,
                "unmatched": unmatched,
                "total_csv_values": len(csv_values),
            }

            self.lookup_issues[csv_col] = unmatched

        return validation_results

    def display_validation_report(self, validation_results: dict):
        """Display validation results"""
        print("\n" + "=" * 80)
        print("🔍 STEP 2: Lookup Table Validation Report")
        print("=" * 80)

        all_valid = True

        for csv_col, result in validation_results.items():
            config = result["lookup_config"]
            matched = result["matched"]
            unmatched = result["unmatched"]
            total = result["total_csv_values"]

            status = "✅" if not unmatched else "⚠️"
            all_valid = all_valid and not unmatched

            print(f"\n{status} Column: '{csv_col}' → {config.table_name}")
            print(f"   Unique values in CSV: {total}")
            print(f"   Matched: {len(matched)}, Unmatched: {len(unmatched)}")

            if unmatched:
                print(f"\n   ❌ UNMATCHED VALUES (need mapping):")
                for val in unmatched[:10]:
                    print(f"      - '{val}'")
                if len(unmatched) > 10:
                    print(f"      ... and {len(unmatched)-10} more")

                print(f"\n   📚 Available values in {config.table_name}:")
                lookup_df = result["lookup_df"]
                for _, row in lookup_df.iterrows():
                    disp_vals = [
                        str(row[c]) for c in config.display_columns if c in row
                    ]
                    id_val = row[config.id_column]
                    print(f"      ID {id_val}: {' | '.join(disp_vals)}")

        if all_valid:
            print("\n" + "=" * 80)
            print("✅ All lookup values validated! Proceed to import.")
            print("=" * 80)
        else:
            print("\n" + "=" * 80)
            print(
                "⚠️ Some values need manual mapping. Run the next cell to create mappings."
            )
            print("=" * 80)

        return all_valid


# Run validation
lookup_validator = LookupValidator(column_mapper.get_mappings(), df)
lookup_validator.load_lookup_data()
lookup_columns = lookup_validator.identify_lookup_columns()

print(f"\n📋 Found {len(lookup_columns)} columns that reference lookup tables:")
for lc in lookup_columns:
    print(f"   - {lc['csv_column']} → {lc['lookup_config'].table_name}")

validation_results = lookup_validator.validate_values(lookup_columns)
all_valid = lookup_validator.display_validation_report(validation_results)

/var/folders/9j/h625g8x90hd0_p1jpflmxd3h0000gn/T/ipykernel_24046/160147562.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  self.lookup_data[lookup_key] = pd.read_sql(query, conn)


✓ Loaded 11 lookup tables

📋 Found 1 columns that reference lookup tables:
   - species → shared.Species

🔍 STEP 2: Lookup Table Validation Report

⚠️ Column: 'species' → shared.Species
   Unique values in CSV: 9
   Matched: 0, Unmatched: 9

   ❌ UNMATCHED VALUES (need mapping):
      - 'Douglas Fir'
      - 'Beech'
      - 'Spruce'
      - 'Silver Fir'
      - 'Oak'
      - 'Larch'
      - 'BE'
      - 'other'
      - 'DF'

   📚 Available values in shared.Species:


KeyError: 'SpeciesID'

In [ ]:
# ============================================================================
# STEP 3: Interactive Value Mapping for Unmatched Lookups
# ============================================================================


class InteractiveValueMapper:
    """Interactive widget for mapping unmatched CSV values to database lookup values"""

    def __init__(self, validation_results: dict, lookup_validator: LookupValidator):
        self.validation_results = validation_results
        self.lookup_validator = lookup_validator
        self.value_mappings = {}  # csv_col -> {csv_value: db_id}
        self.output = widgets.Output()
        self.all_dropdowns = {}

    def display(self):
        """Display interactive mapping interface for unmatched values"""
        print("\n" + "=" * 80)
        print("🔧 STEP 3: Interactive Value Mapping")
        print("=" * 80)
        print("Map unmatched CSV values to database lookup values.\n")
        print("Options:")
        print("  • Select an existing database value to map to")
        print("  • Select '-- CREATE NEW --' to add a new entry to the lookup table")
        print("  • Select '-- SKIP ROW --' to exclude rows with this value\n")

        has_unmatched = False
        all_widgets = []

        for csv_col, result in self.validation_results.items():
            unmatched = result["unmatched"]
            if not unmatched:
                continue

            has_unmatched = True
            config = result["lookup_config"]
            lookup_df = result["lookup_df"]

            # Build options list
            options = ["-- SELECT MAPPING --", "-- CREATE NEW --", "-- SKIP ROW --"]
            for _, row in lookup_df.iterrows():
                id_val = row[config.id_column]
                disp_vals = [str(row[c]) for c in config.display_columns if c in row]
                options.append(f"{id_val}: {' | '.join(disp_vals)}")

            # Section header
            header = widgets.HTML(
                value=f"<h4>📋 Column: <code>{csv_col}</code> → {config.table_name}</h4>"
            )
            all_widgets.append(header)

            self.all_dropdowns[csv_col] = {}

            for csv_val in unmatched:
                # Count occurrences
                count = (self.lookup_validator.csv_df[csv_col] == csv_val).sum()

                label = widgets.HTML(
                    value=f"<code>'{csv_val}'</code> <small>({count} rows)</small>",
                    layout=widgets.Layout(width="300px"),
                )

                dropdown = widgets.Dropdown(
                    options=options,
                    value="-- SELECT MAPPING --",
                    layout=widgets.Layout(width="400px"),
                )

                self.all_dropdowns[csv_col][csv_val] = dropdown

                row = widgets.HBox([label, widgets.HTML("→"), dropdown])
                all_widgets.append(row)

            all_widgets.append(widgets.HTML("<hr>"))

        if not has_unmatched:
            print("✅ No unmatched values! All lookups validated.")
            return

        # Confirm button
        confirm_btn = widgets.Button(
            description="✓ Confirm Value Mappings",
            button_style="success",
            layout=widgets.Layout(width="250px", margin="20px 0"),
        )
        confirm_btn.on_click(self._on_confirm)

        all_widgets.append(confirm_btn)
        all_widgets.append(self.output)

        display(widgets.VBox(all_widgets))

    def _on_confirm(self, btn):
        """Handle confirm button click"""
        with self.output:
            clear_output()

            self.value_mappings = {}
            create_new = []
            skip_values = []
            incomplete = []

            for csv_col, dropdowns in self.all_dropdowns.items():
                self.value_mappings[csv_col] = {}

                for csv_val, dropdown in dropdowns.items():
                    selection = dropdown.value

                    if selection == "-- SELECT MAPPING --":
                        incomplete.append(f"{csv_col}: '{csv_val}'")
                    elif selection == "-- CREATE NEW --":
                        create_new.append(f"{csv_col}: '{csv_val}'")
                        self.value_mappings[csv_col][csv_val] = {
                            "action": "create",
                            "value": csv_val,
                        }
                    elif selection == "-- SKIP ROW --":
                        skip_values.append(f"{csv_col}: '{csv_val}'")
                        self.value_mappings[csv_col][csv_val] = {"action": "skip"}
                    else:
                        # Extract ID from selection "ID: display_value"
                        db_id = int(selection.split(":")[0])
                        self.value_mappings[csv_col][csv_val] = {
                            "action": "map",
                            "id": db_id,
                        }

            print("\n" + "-" * 40)
            print("📊 Value Mapping Summary:")
            print("-" * 40)

            if incomplete:
                print(f"\n⚠️ INCOMPLETE ({len(incomplete)} values not mapped):")
                for item in incomplete:
                    print(f"   - {item}")
                print("\n   Please select a mapping for all values.")
                return

            # Count mapped values
            mapped_count = sum(
                1
                for col_maps in self.value_mappings.values()
                for m in col_maps.values()
                if m.get("action") == "map"
            )

            print(f"\n✅ Mapped to existing: {mapped_count}")

            if create_new:
                print(f"\n🆕 To be created ({len(create_new)}):")
                for item in create_new:
                    print(f"   - {item}")

            if skip_values:
                print(f"\n⏭️ Rows to skip ({len(skip_values)}):")
                for item in skip_values:
                    print(f"   - {item}")

            print("\n" + "=" * 40)
            print("✓ Value mappings confirmed!")
            print("Run the next cell to create any new lookup entries.")
            print("=" * 40)

    def get_value_mappings(self) -> dict:
        """Return the confirmed value mappings"""
        return self.value_mappings


# Create and display value mapper
value_mapper = InteractiveValueMapper(validation_results, lookup_validator)
value_mapper.display()

In [ ]:
# ============================================================================
# STEP 4: Create New Lookup Entries (if needed)
# ============================================================================


class LookupEntryCreator:
    """Creates new entries in lookup tables for unmatched values"""

    def __init__(self, value_mappings: dict, validation_results: dict):
        self.value_mappings = value_mappings
        self.validation_results = validation_results
        self.created_entries = {}  # csv_col -> {csv_value: new_id}
        self.output = widgets.Output()
        self.entry_inputs = {}

    def get_entries_to_create(self) -> list:
        """Get list of entries that need to be created"""
        to_create = []
        for csv_col, mappings in self.value_mappings.items():
            for csv_val, mapping in mappings.items():
                if mapping.get("action") == "create":
                    config = self.validation_results[csv_col]["lookup_config"]
                    to_create.append(
                        {"csv_col": csv_col, "csv_val": csv_val, "config": config}
                    )
        return to_create

    def display(self):
        """Display interface for creating new lookup entries"""
        to_create = self.get_entries_to_create()

        print("\n" + "=" * 80)
        print("🆕 STEP 4: Create New Lookup Entries")
        print("=" * 80)

        if not to_create:
            print(
                "\n✅ No new entries to create! All values mapped to existing entries."
            )
            return

        print(f"\nCreating {len(to_create)} new lookup entries.\n")
        print("Fill in the required fields for each new entry:\n")

        all_widgets = []

        for entry in to_create:
            csv_col = entry["csv_col"]
            csv_val = entry["csv_val"]
            config = entry["config"]

            header = widgets.HTML(
                value=f"<h4>New entry for <code>'{csv_val}'</code> in {config.table_name}</h4>"
            )
            all_widgets.append(header)

            key = f"{csv_col}::{csv_val}"
            self.entry_inputs[key] = {}

            for disp_col in config.display_columns:
                label = widgets.Label(f"{disp_col}:")
                # Pre-fill with CSV value as suggestion
                text_input = widgets.Text(
                    value=csv_val,
                    placeholder=f"Enter {disp_col}",
                    layout=widgets.Layout(width="400px"),
                )
                self.entry_inputs[key][disp_col] = text_input

                row = widgets.HBox([label, text_input])
                all_widgets.append(row)

            all_widgets.append(widgets.HTML("<hr>"))

        # Create button
        create_btn = widgets.Button(
            description="✓ Create Entries in Database",
            button_style="warning",
            layout=widgets.Layout(width="280px", margin="20px 0"),
        )
        create_btn.on_click(self._on_create)

        all_widgets.append(create_btn)
        all_widgets.append(self.output)

        display(widgets.VBox(all_widgets))

    def _on_create(self, btn):
        """Handle create button click"""
        with self.output:
            clear_output()

            to_create = self.get_entries_to_create()

            conn = psycopg2.connect(
                host=POSTGRES_HOST,
                user=POSTGRES_USER_POOLER,
                password=POSTGRES_PASSWORD,
                database=POSTGRES_DATABASE,
                port=POSTGRES_PORT,
            )
            cursor = conn.cursor()

            print("\n" + "-" * 40)
            print("Creating new lookup entries...")
            print("-" * 40)

            for entry in to_create:
                csv_col = entry["csv_col"]
                csv_val = entry["csv_val"]
                config = entry["config"]
                key = f"{csv_col}::{csv_val}"

                # Get values from inputs
                values = {}
                for disp_col, text_input in self.entry_inputs[key].items():
                    values[disp_col] = text_input.value

                # Build INSERT query
                columns = list(values.keys())
                placeholders = ["%s"] * len(columns)

                query = f"""
                    INSERT INTO {config.table_name} ({', '.join(columns)})
                    VALUES ({', '.join(placeholders)})
                    RETURNING {config.id_column}
                """

                try:
                    cursor.execute(query, list(values.values()))
                    new_id = cursor.fetchone()[0]

                    if csv_col not in self.created_entries:
                        self.created_entries[csv_col] = {}
                    self.created_entries[csv_col][csv_val] = new_id

                    print(f"✅ Created: {config.table_name} ID={new_id}")
                    print(f"   Values: {values}")

                except Exception as e:
                    print(f"❌ Failed to create entry for '{csv_val}': {e}")
                    conn.rollback()
                    cursor.close()
                    conn.close()
                    return

            conn.commit()
            cursor.close()
            conn.close()

            print("\n" + "=" * 40)
            print(f"✓ Created {len(to_create)} new entries!")
            print("=" * 40)

    def get_created_entries(self) -> dict:
        """Return the created entries with their new IDs"""
        return self.created_entries


# Create and display entry creator
entry_creator = LookupEntryCreator(
    value_mapper.get_value_mappings(), validation_results
)
entry_creator.display()

In [ ]:
# ============================================================================
# STEP 5: Build Final Mapping Configuration
# ============================================================================


def build_final_mapping_config(
    column_mappings: dict,
    value_mappings: dict,
    created_entries: dict,
    validation_results: dict,
) -> dict:
    """
    Consolidate all mapping information into a final configuration
    that can be used for the actual data import.
    """

    final_config = {
        "column_mappings": {},  # csv_col -> db_target
        "lookup_transforms": {},  # csv_col -> {csv_value: db_id}
        "skip_values": {},  # csv_col -> [csv_values_to_skip]
        "coordinate_mappings": {},  # Special handling for coordinates
    }

    for csv_col, db_target in column_mappings.items():
        if db_target.startswith("-- COORDINATE"):
            coord_type = "lat_lon" if "lat/lon" in db_target else "x_y"
            final_config["coordinate_mappings"][csv_col] = coord_type
        elif not db_target.startswith("--"):
            final_config["column_mappings"][csv_col] = db_target

    # Process lookup transforms
    for csv_col, mappings in value_mappings.items():
        transforms = {}
        skips = []

        for csv_val, mapping in mappings.items():
            action = mapping.get("action")

            if action == "map":
                transforms[csv_val] = mapping["id"]
            elif action == "create":
                # Get the new ID from created_entries
                if csv_col in created_entries and csv_val in created_entries[csv_col]:
                    transforms[csv_val] = created_entries[csv_col][csv_val]
            elif action == "skip":
                skips.append(csv_val)

        if transforms:
            final_config["lookup_transforms"][csv_col] = transforms
        if skips:
            final_config["skip_values"][csv_col] = skips

    return final_config


# Build final configuration
final_config = build_final_mapping_config(
    column_mapper.get_mappings(),
    value_mapper.get_value_mappings(),
    entry_creator.get_created_entries(),
    validation_results,
)

print("\n" + "=" * 80)
print("📋 FINAL MAPPING CONFIGURATION")
print("=" * 80)

print("\n📍 Column Mappings (CSV → Database):")
for csv_col, db_target in final_config["column_mappings"].items():
    print(f"   {csv_col} → {db_target}")

if final_config["coordinate_mappings"]:
    print("\n🌍 Coordinate Columns:")
    for csv_col, coord_type in final_config["coordinate_mappings"].items():
        print(f"   {csv_col} ({coord_type})")

if final_config["lookup_transforms"]:
    print("\n🔄 Lookup Value Transforms:")
    for csv_col, transforms in final_config["lookup_transforms"].items():
        print(f"   {csv_col}:")
        for csv_val, db_id in transforms.items():
            print(f"      '{csv_val}' → ID {db_id}")

if final_config["skip_values"]:
    print("\n⏭️ Values to Skip (rows excluded):")
    for csv_col, values in final_config["skip_values"].items():
        print(f"   {csv_col}: {values}")

# Save configuration to JSON
config_path = Path(CSV_PATH).parent / f"{Path(CSV_PATH).stem}_import_config.json"
with open(config_path, "w") as f:
    json.dump(final_config, f, indent=2)
print(f"\n✓ Configuration saved to: {config_path}")

In [ ]:
# ============================================================================
# STEP 6: Transform and Preview Data
# ============================================================================


def transform_data_for_import(csv_df: pd.DataFrame, final_config: dict) -> pd.DataFrame:
    """
    Apply all transformations to prepare data for database import.

    - Maps CSV columns to database columns
    - Transforms lookup values to IDs
    - Filters out rows with skip values
    - Handles coordinate transformations
    """

    # Start with a copy
    transformed_df = csv_df.copy()

    # 1. Filter out rows with skip values
    for csv_col, skip_values in final_config.get("skip_values", {}).items():
        if csv_col in transformed_df.columns:
            mask = ~transformed_df[csv_col].isin(skip_values)
            removed_count = len(transformed_df) - mask.sum()
            transformed_df = transformed_df[mask]
            print(f"⏭️ Removed {removed_count} rows with skip values in '{csv_col}'")

    # 2. Apply lookup transforms (replace CSV values with database IDs)
    for csv_col, transforms in final_config.get("lookup_transforms", {}).items():
        if csv_col in transformed_df.columns:
            # Create a mapping function
            def transform_value(val):
                val_str = str(val).strip() if pd.notna(val) else None
                if val_str in transforms:
                    return transforms[val_str]
                return val  # Keep original if no transform

            transformed_df[csv_col] = transformed_df[csv_col].apply(transform_value)
            print(f"🔄 Transformed lookup values in '{csv_col}'")

    # 3. Handle coordinates
    coord_mappings = final_config.get("coordinate_mappings", {})
    if coord_mappings:
        # Detect coordinate column pairs
        col_names_lower = {col.lower(): col for col in transformed_df.columns}

        lat_col = next(
            (
                col_names_lower[p]
                for p in ["latitude", "lat", "gps_latitude", "gps_lat"]
                if p in col_names_lower
            ),
            None,
        )
        lon_col = next(
            (
                col_names_lower[p]
                for p in ["longitude", "lon", "gps_longitude", "gps_lon"]
                if p in col_names_lower
            ),
            None,
        )

        if lat_col and lon_col:
            # Create WKT geometry string
            def create_point_wkt(row):
                lat = row[lat_col]
                lon = row[lon_col]
                if pd.notna(lat) and pd.notna(lon):
                    return f"SRID=4326;POINT({lon} {lat})"
                return None

            transformed_df["_position_wkt"] = transformed_df.apply(
                create_point_wkt, axis=1
            )
            print(f"🌍 Created geometry from '{lat_col}' and '{lon_col}'")

    # 4. Rename columns according to mapping
    column_renames = {}
    for csv_col, db_target in final_config.get("column_mappings", {}).items():
        if csv_col in transformed_df.columns:
            # Extract just the column name from schema.table.column
            parts = db_target.split(".")
            db_col = parts[-1]  # Last part is the column name
            column_renames[csv_col] = db_col

    # Don't rename yet - keep original for preview

    return transformed_df


# Transform data
print("\n" + "=" * 80)
print("🔄 STEP 6: Data Transformation Preview")
print("=" * 80)

transformed_df = transform_data_for_import(df, final_config)

print(f"\n📊 Data Summary:")
print(f"   Original rows: {len(df)}")
print(f"   After filtering: {len(transformed_df)}")
print(f"   Columns: {len(transformed_df.columns)}")

# Show preview of transformed data
print("\n📋 Preview of transformed data (first 5 rows):")

# Select only mapped columns for display
mapped_cols = list(final_config.get("column_mappings", {}).keys())
if "_position_wkt" in transformed_df.columns:
    mapped_cols.append("_position_wkt")

if mapped_cols:
    preview_cols = [c for c in mapped_cols if c in transformed_df.columns]
    display(transformed_df[preview_cols].head())

## Step 7: Execute Import

⚠️ **Warning**: This will insert data into the database. Review the preview above before proceeding.

In [ ]:
# ============================================================================
# STEP 7: Execute Database Import
# ============================================================================


def execute_import(
    transformed_df: pd.DataFrame, final_config: dict, dry_run: bool = True
) -> dict:
    """
    Execute the actual database import.

    Args:
        transformed_df: The transformed DataFrame ready for import
        final_config: The mapping configuration
        dry_run: If True, rollback after testing (no actual changes)

    Returns:
        dict with import statistics
    """

    stats = {"rows_attempted": 0, "rows_inserted": 0, "rows_failed": 0, "errors": []}

    # Determine target table from column mappings
    target_tables = set()
    column_map = {}  # csv_col -> (schema, table, column)

    for csv_col, db_target in final_config.get("column_mappings", {}).items():
        parts = db_target.split(".")
        if len(parts) == 3:
            schema, table, column = parts
            target_tables.add(f"{schema}.{table}")
            column_map[csv_col] = (schema, table, column)

    if len(target_tables) > 1:
        print(f"⚠️ Multiple target tables detected: {target_tables}")
        print("   This import supports one target table at a time.")
        return stats

    if not target_tables:
        print("❌ No valid column mappings found.")
        return stats

    target_table = list(target_tables)[0]
    print(f"🎯 Target table: {target_table}")

    # Build column name mapping (csv -> db column name)
    db_columns = []
    csv_columns = []

    for csv_col, (schema, table, column) in column_map.items():
        if csv_col in transformed_df.columns:
            csv_columns.append(csv_col)
            db_columns.append(column)

    # Add position if we have coordinates
    if "_position_wkt" in transformed_df.columns:
        csv_columns.append("_position_wkt")
        db_columns.append("Position")

    print(f"📋 Columns to insert: {db_columns}")

    # Connect to database
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        user=POSTGRES_USER_POOLER,
        password=POSTGRES_PASSWORD,
        database=POSTGRES_DATABASE,
        port=POSTGRES_PORT,
    )
    cursor = conn.cursor()

    # Build INSERT query
    placeholders = []
    for i, db_col in enumerate(db_columns):
        if db_col == "Position":
            placeholders.append("ST_GeomFromEWKT(%s)")
        else:
            placeholders.append("%s")

    insert_query = f"""
        INSERT INTO {target_table} ({', '.join(db_columns)})
        VALUES ({', '.join(placeholders)})
    """

    print(
        f"\n{'🔍 DRY RUN - Testing without commit' if dry_run else '⚡ LIVE RUN - Inserting data'}"
    )
    print("-" * 40)

    # Insert rows
    for idx, row in transformed_df.iterrows():
        stats["rows_attempted"] += 1

        values = []
        for csv_col in csv_columns:
            val = row[csv_col]
            if pd.isna(val):
                values.append(None)
            else:
                values.append(val)

        try:
            cursor.execute(insert_query, values)
            stats["rows_inserted"] += 1
        except Exception as e:
            stats["rows_failed"] += 1
            if len(stats["errors"]) < 5:  # Only keep first 5 errors
                stats["errors"].append(f"Row {idx}: {str(e)[:100]}")

    if dry_run:
        conn.rollback()
        print("🔄 Rolled back (dry run)")
    else:
        conn.commit()
        print("✅ Committed to database")

    cursor.close()
    conn.close()

    return stats


# Execute import (DRY RUN first)
print("\n" + "=" * 80)
print("🚀 STEP 7: Execute Import")
print("=" * 80)

# Toggle this to False for actual import
DRY_RUN = True

stats = execute_import(transformed_df, final_config, dry_run=DRY_RUN)

print("\n" + "=" * 40)
print("📊 Import Statistics:")
print("=" * 40)
print(f"   Rows attempted: {stats['rows_attempted']}")
print(f"   Rows inserted:  {stats['rows_inserted']}")
print(f"   Rows failed:    {stats['rows_failed']}")

if stats["errors"]:
    print("\n❌ Sample errors:")
    for err in stats["errors"]:
        print(f"   {err}")

if DRY_RUN:
    print("\n" + "=" * 40)
    print("ℹ️  This was a DRY RUN. No data was actually inserted.")
    print("    Set DRY_RUN = False and re-run to perform actual import.")
    print("=" * 40)

## Utility: Load Saved Configuration

If you've previously saved a mapping configuration, you can load it here to skip the interactive mapping steps.

In [ ]:
# ============================================================================
# UTILITY: Load Saved Configuration
# ============================================================================


def load_saved_config(config_path: str) -> dict:
    """Load a previously saved import configuration"""
    with open(config_path, "r") as f:
        config = json.load(f)

    print(f"✓ Loaded configuration from: {config_path}")
    print(f"\n📋 Column Mappings: {len(config.get('column_mappings', {}))}")
    print(f"🔄 Lookup Transforms: {len(config.get('lookup_transforms', {}))}")
    print(f"⏭️ Skip Values: {len(config.get('skip_values', {}))}")

    return config


# Example: Load a saved configuration
# Uncomment and modify the path to use:

# saved_config_path = "../data/mathisle_250904_import_config.json"
# if Path(saved_config_path).exists():
#     final_config = load_saved_config(saved_config_path)
#     transformed_df = transform_data_for_import(df, final_config)
# else:
#     print(f"Config file not found: {saved_config_path}")

print(
    "To load a saved config, uncomment the code above and specify your config file path."
)

---

# 📂 Data Loading Section

**Run these cells BEFORE the Interactive Data Import Wizard above!**

## Load CSV File

In [10]:
# Specify your CSV file path
CSV_PATH = "../data/ecosense_250911.csv"  # Change this to your CSV path

# Load CSV
if not Path(CSV_PATH).exists():
    print(f"❌ CSV file not found: {CSV_PATH}")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"✓ Loaded CSV: {Path(CSV_PATH).name}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {', '.join(df.columns)}")

✓ Loaded CSV: ecosense_250911.csv
  Rows: 1530
  Columns: Unnamed: 0, species, qr_code_id, tree_image, comment, odk_KEY, x_32632, y_32632, diameter_m, tls_treeheight, plot_id, tree_id, full_id, elevation, sensor_tree


In [11]:
# Display first few rows
print(f"\nFirst 3 rows of {Path(CSV_PATH).name}:")
display(df.head(3))


First 3 rows of ecosense_250911.csv:


,Unnamed: 0,species,qr_code_id,tree_image,comment,odk_KEY,x_32632,y_32632,diameter_m,tls_treeheight,plot_id,tree_id,full_id,elevation,sensor_tree
0,45,Douglas Fir,https://dt.unr.uni-freiburg.de/ecosense/8_15,1732627753259.jpg,NaN,uuid:e209a154-c01c-4534-b144-cfa7d7713b22/loop...,416739.174632,5.346747e+06,0.544310,NaN,8.0,15.0,8_15,520.677063,True
1,46,Beech,https://dt.unr.uni-freiburg.de/ecosense/8_16,1732627829838.jpg,NaN,uuid:e209a154-c01c-4534-b144-cfa7d7713b22/loop...,416741.364569,5.346744e+06,0.210085,NaN,8.0,16.0,8_16,520.599854,True
2,47,Douglas Fir,https://dt.unr.uni-freiburg.de/ecosense/8_27,1732628005341.jpg,NaN,uuid:e209a154-c01c-4534-b144-cfa7d7713b22/loop...,416744.575059,5.346743e+06,0.439268,NaN,8.0,27.0,8_27,520.770813,True


---

# 📖 Reference: Coordinate Mapping Guide

*This section provides reference information for understanding coordinate handling. The Interactive Import Wizard above handles this automatically.*

If your CSV has separate lat/lon or x/y columns, read this section to understand mapping options.

In [ ]:
print(
    """
="*80
📍 COORDINATE MAPPING - How to handle spatial data
="*80

Your CSV has coordinates in separate columns (latitude/longitude or x/y).
The database stores them as a single geometry column (Position).

MAPPING STRATEGIES:

1️⃣  COMBINED COLUMN MAPPING (Recommended)
   Format: lat_lon:EPSG_CODE or x_y:EPSG_CODE

   Examples:
   - For lat/lon columns: 'coordinates' maps to: lat_lon:EPSG:4326
   - For x/y columns: 'utm_coords' maps to: x_y:EPSG:32632

   Benefits: Automatic combination + CRS transformation

   Available column names:
   - Latitude: latitude, lat, lat_col, latitude_col, gps_latitude
   - Longitude: longitude, lon, lon_col, longitude_col, gps_longitude
   - X/Easting: x, x_col, easting, easting_col, utm_x, x_32632
   - Y/Northing: y, y_col, northing, northing_col, utm_y, y_32632

2️⃣  MANUAL GEOMETRY STRING
   Map to: shared.Locations.Position (or trees.Trees.Position, etc.)
   Create WKT format before import: POINT(lon lat)

3️⃣  SEPARATE COORDINATES (Not Recommended)
   ❌ Don't map lat/lon to separate database columns
   ✅ Combine them first using option 1 or 2

CRS SUPPORT:
   - EPSG:4326 (WGS84) - default, no transformation needed
   - EPSG:32632 (UTM Zone 32N) - automatically transforms to WGS84
   - Other EPSG codes - specify in mapping format

COMMON COORDINATES IN YOUR DATA:
   Mathisle: gps_latitude, gps_longitude (WGS84)
   EcoSense: x_32632, y_32632 (UTM Zone 32N)
"""
)

---

# 📜 Legacy: Manual Column Mapping (Alternative Approach)

*Use this section only if you prefer manual JSON-based mapping instead of the Interactive Import Wizard.*

## Define Column Mapping (Manual)

In [ ]:
# Define your column mapping here
# Format: csv_column: "schema.table.column" or "lat_lon:EPSG:4326" or "x_y:EPSG:32632" or "SKIP"

mapping = {
    # Example for Mathisle trees CSV:
    # "species_short": "shared.Species.CommonName",
    # "gps_latitude": "lat_lon:EPSG:4326",  # Auto-detects gps_longitude
    # "gps_longitude": "SKIP",  # Skip since lat_lon mapping handles it
    # "DBH": "trees.Trees.FieldNotes",
    # "TreeID": "trees.Trees.FieldNotes",
    # "height": "trees.Trees.Height_m",
}

print("Mapping defined. Edit the cell above to configure for your CSV.")
print(f"\nCSV Columns: {list(df.columns)}")
print(f"\nCurrent Mapping:")
for csv_col, target in mapping.items():
    print(f"  '{csv_col}' -> {target}")

### Use LOOKUP to inspect CSV values before deciding on mapping

Edit the cell below to inspect specific columns from your CSV

In [ ]:
# Inspect a column to understand its values
column_to_inspect = "species_short"  # Change this to any column name

if column_to_inspect in df.columns:
    unique_values = df[column_to_inspect].unique()
    print(f"\nSample values from '{column_to_inspect}' ({len(unique_values)} unique):")
    for val in unique_values[:10]:
        print(f"  - {val}")
    if len(unique_values) > 10:
        print(f"  ... and {len(unique_values) - 10} more")
else:
    print(f"Column '{column_to_inspect}' not found in CSV")
    print(f"Available columns: {list(df.columns)}")

## Step 7: Save Mapping as JSON

In [ ]:
# Save mapping to JSON for reuse
mapping_path = Path(CSV_PATH).parent / f"{Path(CSV_PATH).stem}_mapping.json"

with open(mapping_path, "w") as f:
    json.dump(mapping, f, indent=2)

print(f"✓ Mapping saved to: {mapping_path}")

# Display saved mapping
with open(mapping_path, "r") as f:
    saved_mapping = json.load(f)
    print(f"\nSaved mapping:")
    print(json.dumps(saved_mapping, indent=2))

## Step 8: Process Coordinates

In [ ]:
def detect_coordinate_columns(df, format_spec):
    """Detect lat/lon or x/y columns by flexible naming"""
    col_names_lower = {col.lower(): col for col in df.columns}

    if format_spec.startswith("lat_lon"):
        # Find latitude and longitude
        lat_patterns = ["latitude", "lat", "gps_latitude", "gps_lat"]
        lon_patterns = ["longitude", "lon", "gps_longitude", "gps_lon"]

        lat_col = next(
            (col_names_lower[p] for p in lat_patterns if p in col_names_lower), None
        )
        lon_col = next(
            (col_names_lower[p] for p in lon_patterns if p in col_names_lower), None
        )

        return {"lat": lat_col, "lon": lon_col}

    elif format_spec.startswith("x_y"):
        # Find x and y
        x_patterns = ["x", "easting", "utm_x", "x_32632", "x_32633"]
        y_patterns = ["y", "northing", "utm_y", "y_32632", "y_32633"]

        x_col = next(
            (col_names_lower[p] for p in x_patterns if p in col_names_lower), None
        )
        y_col = next(
            (col_names_lower[p] for p in y_patterns if p in col_names_lower), None
        )

        return {"x": x_col, "y": y_col}

    return {}


def process_coordinates(df, mapping):
    """Process coordinate columns and create Position geometry"""
    if not HAS_PYPROJ:
        print("⚠️  pyproj not available. Coordinate processing skipped.")
        return df

    for csv_col, target in mapping.items():
        if not isinstance(target, str) or ":" not in target:
            continue

        coord_type, crs = target.split(":", 1)

        if coord_type == "lat_lon":
            coords = detect_coordinate_columns(df, target)
            if not coords["lat"] or not coords["lon"]:
                print(f"⚠️  Could not find lat/lon columns")
                continue

            lat_col = coords["lat"]
            lon_col = coords["lon"]

            print(f"📍 Processing {csv_col}: lat={lat_col}, lon={lon_col}, CRS={crs}")

            # Transform if needed
            if crs.upper() != "EPSG:4326":
                transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

                def transform_latlon(row):
                    if pd.notna(row[lat_col]) and pd.notna(row[lon_col]):
                        lon_transformed, lat_transformed = transformer.transform(
                            row[lon_col], row[lat_col]
                        )
                        return f"POINT({lon_transformed} {lat_transformed})"
                    return None

                df["Position"] = df.apply(transform_latlon, axis=1)
            else:
                df["Position"] = df.apply(
                    lambda row: (
                        f"POINT({row[lon_col]} {row[lat_col]})"
                        if pd.notna(row[lat_col]) and pd.notna(row[lon_col])
                        else None
                    ),
                    axis=1,
                )

            valid_count = df["Position"].notna().sum()
            print(f"✓ Created {valid_count} Position geometries")

        elif coord_type == "x_y":
            coords = detect_coordinate_columns(df, target)
            if not coords["x"] or not coords["y"]:
                print(f"⚠️  Could not find x/y columns")
                continue

            x_col = coords["x"]
            y_col = coords["y"]

            print(f"📍 Processing {csv_col}: x={x_col}, y={y_col}, CRS={crs}")

            # Transform to WGS84
            transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

            def transform_xy(row):
                if pd.notna(row[x_col]) and pd.notna(row[y_col]):
                    lon_transformed, lat_transformed = transformer.transform(
                        row[x_col], row[y_col]
                    )
                    return f"POINT({lon_transformed} {lat_transformed})"
                return None

            df["Position"] = df.apply(transform_xy, axis=1)

            valid_count = df["Position"].notna().sum()
            print(f"✓ Transformed to WGS84")
            print(f"✓ Created {valid_count} Position geometries")

    return df


# Process coordinates if any are defined
df = process_coordinates(df, mapping)

## Step 9: Organize Data by Table

In [ ]:
def apply_mapping(df, mapping):
    """Organize data by target table"""
    table_dfs = {}

    for csv_col, target in mapping.items():
        if target is None or ":" in str(target):  # Skip special formats
            continue

        try:
            schema, table, column = target.split(".")
            table_key = f"{schema}.{table}"

            if table_key not in table_dfs:
                table_dfs[table_key] = pd.DataFrame()

            table_dfs[table_key][column] = df[csv_col]
        except ValueError:
            print(f"⚠️  Invalid mapping for '{csv_col}': {target}")

    # Add Position column if it exists
    if "Position" in df.columns:
        if "trees.Trees" in table_dfs:
            table_dfs["trees.Trees"]["Position"] = df["Position"]
        elif "sensor.Sensors" in table_dfs:
            table_dfs["sensor.Sensors"]["Position"] = df["Position"]
        elif len(table_dfs) > 0:
            first_table = list(table_dfs.keys())[0]
            table_dfs[first_table]["Position"] = df["Position"]

    return table_dfs


# Apply mapping
table_dfs = apply_mapping(df, mapping)
print(f"✓ Organized data into {len(table_dfs)} tables")

## Step 10: Preview Data

In [ ]:
# Display preview of data for each table
print("\n" + "=" * 80)
print("DATA PREVIEW - How data will be inserted into each table")
print("=" * 80)

for table_name, table_df in table_dfs.items():
    print(f"\n📊 {table_name} ({len(table_df)} rows, {len(table_df.columns)} columns)")
    print(f"   Columns: {', '.join(table_df.columns)}")
    print(f"\n   First 2 rows:")
    display(table_df.head(2))

## Step 11: Insert Data into Database (Optional)

Uncomment and run this cell to insert the prepared data into the database.

In [ ]:
# This is a template for inserting data
# Uncomment the code below and run to insert

"""
import psycopg2
from psycopg2.extras import execute_values

def insert_table_data(table_name, df):
    '''Insert DataFrame into specified table'''
    schema, table = table_name.split('.')

    try:
        conn = psycopg2.connect(
            host=POSTGRES_HOST,
            user=POSTGRES_USER_POOLER,
            password=POSTGRES_PASSWORD,
            database=POSTGRES_DATABASE,
            port=POSTGRES_PORT
        )
        cur = conn.cursor()

        # Prepare INSERT statement
        columns = ', '.join(df.columns)
        placeholders = ', '.join(['%s'] * len(df.columns))
        query = f"INSERT INTO {schema}.{table} ({columns}) VALUES ({placeholders})"

        # Convert DataFrame to list of tuples
        values = [tuple(row) for row in df.itertuples(index=False, name=None)]

        # Execute insert
        execute_values(cur, query, values)
        conn.commit()

        print(f"✓ Inserted {len(df)} rows into {table_name}")

        conn.close()
    except Exception as e:
        print(f"❌ Failed to insert into {table_name}: {e}")

# Insert all tables
for table_name, table_df in table_dfs.items():
    if len(table_df) > 0:
        insert_table_data(table_name, table_df)
"""

## Summary

✓ **What you've done:**
1. Connected to the Digital Forest Twin database
2. Introspected available database schema
3. Loaded reference data (Species, Locations, SensorTypes)
4. Loaded your CSV file
5. Defined column mappings
6. Processed coordinates (lat/lon or x/y)
7. Organized data by table
8. Previewed data before insertion

**Next steps:**
1. Review the data preview above
2. Mapping is saved to: `{mapping_path}`
3. Data is ready in the `table_dfs` variable for insertion
4. Uncomment and run Step 11 to insert into database
5. Or use the data in any other way you need

**Tips:**
- You can reuse saved mappings by loading the JSON file
- Use the LOOKUP feature (Step 6) to inspect column values
- Modify the mapping cell to change column assignments
- Run cells individually to test each step